In [1]:
import math

import ee
import geemap

import pandas as pd
import csv
import time
import os

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
project_id = "ac215-bhargav"
ee.Authenticate()
ee.Initialize(project=project_id)

# CODE

## COMMON FUNCTIONS

In [4]:
def build_points_from_group(group_df):
    point_features = []
    for _, row in group_df.iterrows():
        lon    = pd.to_numeric(row['longitude'], errors='coerce')
        lat    = pd.to_numeric(row['latitude'],  errors='coerce')
        row_id = int(row['row_id'])

        if pd.isna(lon) or pd.isna(lat):
            print(f"Skipping invalid coordinate: row_id={row_id}, lon={lon}, lat={lat}")
            continue

        point_features.append(
            ee.Feature(ee.Geometry.Point([lon, lat])).set({
                'row_id'  : row_id,
                'orig_lon': lon,
                'orig_lat': lat
            })
        )

    print(f"Valid points: {len(point_features)} / {len(group_df)}")
    return ee.FeatureCollection(point_features)

## SIMARD FUNCTIONS

In [5]:
def extract_simard_canopy_height(points):
    """
    Extract Simard et al. mangrove canopy height at each point location.
    Dataset : NASA/JPL/global_forest_canopy_height_2005
    Band    : '1' = canopy height in meters
    Resolution: 30m
    """
    simard = ee.Image("NASA/JPL/global_forest_canopy_height_2005")

    reduced = simard.select('1').reduceRegions(
        collection = points,
        reducer    = ee.Reducer.mean(),
        scale      = 30
    )

    def add_metadata(feature):
        return feature.set({
            'row_id'  : feature.get('row_id'),
            'orig_lon': feature.get('orig_lon'),
            'orig_lat': feature.get('orig_lat')
        })

    return reduced.map(add_metadata)

In [6]:
def create_simard_df(points, max_elements=4500):
    print("EXTRACTING SIMARD CANOPY HEIGHT")

    n_points    = points.size().getInfo()
    points_list = points.toList(n_points)
    all_rows    = []

    for start in range(0, n_points, max_elements):
        end         = min(start + max_elements, n_points)
        point_chunk = ee.FeatureCollection(points_list.slice(start, end))

        result = extract_simard_canopy_height(point_chunk)
        info   = result.getInfo()

        for feature in info['features']:
            props = feature['properties']
            all_rows.append({
                'row_id'         : props.get('row_id'),
                'longitude'      : props.get('orig_lon'),
                'latitude'       : props.get('orig_lat'),
                'simard_height_m': props.get('mean')
            })

        print(f"  Processed {end}/{n_points} points")

    df = pd.DataFrame(all_rows)
    print(f"Extracted rows    : {len(df)}")
    print(f"Null height values: {df['simard_height_m'].isnull().sum()}")
    if df['simard_height_m'].notna().any():
        print(f"Height min        : {df['simard_height_m'].min():.2f} m")
        print(f"Height max        : {df['simard_height_m'].max():.2f} m")
        print(f"Height mean       : {df['simard_height_m'].mean():.2f} m")
    return df

In [18]:
def run_simard(emit_df, max_elements=4500):
    # Load AGB data
    #agb_df              = pd.read_csv(input_csv)
    agb_df              = emit_df.reset_index(drop=True)
    agb_df['row_id']    = agb_df.index.astype(int)
    #agb_df['latitude']  = pd.to_numeric(agb_df['latitude'],  errors='coerce')
    #agb_df['longitude'] = pd.to_numeric(agb_df['longitude'], errors='coerce')
    #agb_df              = agb_df.dropna(subset=['latitude', 'longitude'])

    print(f"Total rows    : {len(agb_df)}")
    print(f"Unique plots  : {agb_df['plot_id'].nunique()}")

    # Build GEE points
    points     = build_points_from_group(agb_df)

    # Extract Simard height
    simard_df  = create_simard_df(points, max_elements)

    return simard_df

In [19]:
def extract_simard(df):
    simard_df = run_simard(df)
    
    simard_dupes = simard_df.groupby(['latitude', 'longitude']).size()
    #print(f"Max rows per coordinate in simard_df : {simard_dupes.max()}")
    #print(f"Coordinates with duplicates          : {(simard_dupes > 1).sum()}")
    print(simard_dupes[simard_dupes > 1].head(10))
    
    simard_deduped = simard_df.groupby(
        ['latitude', 'longitude'], as_index=False
    )['simard_height_m'].mean()

    return simard_deduped

In [20]:
def merge_simard_dfs(source_df, target_df):
    merged_df = source_df.merge(
        target_df[['longitude', 'latitude', 'simard_height_m']],
        on  = ['longitude', 'latitude'],
        how = 'left'
    )
    
    return merged_df

## TANDEM-X FUNCTIONS

In [21]:
def extract_tandemx_canopy_height(points, buffer_meters=30):
    """
    Extract TanDEM-X mangrove canopy height at each point location.
    Dataset : projects/sat-io/open-datasets/GLOBAL_MANGROVE_HT_TANDEMX
    Resolution: 12m
    Non-mangrove pixels = 0
    """
    # Mosaic the image collection into a single image
    tandemx = ee.ImageCollection(
        "projects/sat-io/open-datasets/GLOBAL_MANGROVE_HT_TANDEMX"
    ).mosaic()

    # Buffer points to capture surrounding pixels
    buffered_points = points.map(
        lambda f: f.buffer(buffer_meters).copyProperties(f)
    )

    # Extract mean height within buffer
    reduced = tandemx.reduceRegions(
        collection = buffered_points,
        reducer    = ee.Reducer.mean(),
        scale      = 12
    )

    def add_metadata(feature):
        return feature.set({
            'row_id'  : feature.get('row_id'),
            'orig_lon': feature.get('orig_lon'),
            'orig_lat': feature.get('orig_lat')
        })

    return reduced.map(add_metadata)

In [22]:
def create_tandemx_df(points, max_elements=4500):
    print("EXTRACTING TANDEMX CANOPY HEIGHT")

    n_points    = points.size().getInfo()
    points_list = points.toList(n_points)
    all_rows    = []

    for start in range(0, n_points, max_elements):
        end         = min(start + max_elements, n_points)
        point_chunk = ee.FeatureCollection(points_list.slice(start, end))

        result = extract_tandemx_canopy_height(point_chunk)
        info   = result.getInfo()

        for feature in info['features']:
            props  = feature['properties']
            height = props.get('mean')
            if height == 0:
                height = None
            all_rows.append({
                'longitude'       : props.get('orig_lon'),
                'latitude'        : props.get('orig_lat'),
                'tandemx_height_m': height
            })

        print(f"  Processed {end}/{n_points} points")

    df = pd.DataFrame(all_rows)
    print(f"Extracted rows         : {len(df)}")
    print(f"Null height values     : {df['tandemx_height_m'].isnull().sum()}")
    if df['tandemx_height_m'].notna().any():
        print(f"Height min             : {df['tandemx_height_m'].min():.2f} m")
        print(f"Height max             : {df['tandemx_height_m'].max():.2f} m")
        print(f"Height mean            : {df['tandemx_height_m'].mean():.2f} m")
        print(f"Unique height values   : {df['tandemx_height_m'].nunique()}")
    return df

In [23]:
def run_tandemx(emit_df, max_elements=4500):
    # Load AGB data
    #agb_df              = pd.read_csv(input_csv)
    agb_df              = emit_df.reset_index(drop=True)
    agb_df['row_id']    = agb_df.index.astype(int)
    #agb_df['latitude']  = pd.to_numeric(agb_df['latitude'],  errors='coerce')
    #agb_df['longitude'] = pd.to_numeric(agb_df['longitude'], errors='coerce')
    #agb_df              = agb_df.dropna(subset=['latitude', 'longitude'])

    print(f"Total rows    : {len(agb_df)}")
    print(f"Unique plots  : {agb_df['plot_id'].nunique()}")

    # Build GEE points
    points     = build_points_from_group(agb_df)

    # Extract Simard height
    tandemx_df  = create_tandemx_df(points, max_elements)

    return tandemx_df

In [24]:
def extract_tandemx(df):
    tandemx_df = run_tandemx(df)
    # Impute missing values
    null_count = tandemx_df['tandemx_height_m'].isnull().sum()
    print(f"TandemX height null count   : {null_count}")
    
    if null_count:
        median_height = tandemx_df['tandemx_height_m'].median()
        tandemx_df['tandemx_height_m'] = tandemx_df['tandemx_height_m'].fillna(median_height)
    print(f"tandemx_df height null count   : {tandemx_df['tandemx_height_m'].isnull().sum()}")

    tandemx_deduped = tandemx_df.groupby(
        ['latitude', 'longitude'], as_index=False
    )['tandemx_height_m'].mean()

    return tandemx_deduped

In [25]:
def merge_tandex_dfs(source_df, target_df):
    merged_df = source_df.merge(
        target_df[['longitude', 'latitude', 'tandemx_height_m']],
        on  = ['longitude', 'latitude'],
        how = 'left'
    )
    return merged_df

# EXTRACT CANOPY HEIGHTS FOR EMIT

In [26]:
INPUT_CSV  = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT.csv"
OUTPUT_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EMIT_EO_CANOPY.csv"

emit_df = pd.read_csv(INPUT_CSV)
#print(emit_df.columns)
print(f"emit_df height null count   : {emit_df['height'].isnull().sum()}")

emit_df height null count   : 0


## Extract SIMARD for EMIT data

In [27]:
emit_simard_df = extract_simard(emit_df)

Total rows    : 3880
Unique plots  : 59
Valid points: 3880 / 3880
EXTRACTING SIMARD CANOPY HEIGHT
  Processed 3880/3880 points
Extracted rows    : 3880
Null height values: 0
Height min        : 0.00 m
Height max        : 14.00 m
Height mean       : 3.97 m
latitude   longitude 
16.182060  -88.656500    105
16.182147  -88.656372    105
16.182230  -88.656190    117
16.182337  -88.656050     70
16.182404  -88.655890     90
16.182480  -88.655700    118
16.288337  -88.591185     81
16.288585  -88.591221    116
16.288840  -88.591300     76
16.289035  -88.591352     95
dtype: int64


## Extract TandemX for EMIT data

In [28]:
emit_tandemx_df = extract_tandemx(emit_df)

Total rows    : 3880
Unique plots  : 59
Valid points: 3880 / 3880
EXTRACTING TANDEMX CANOPY HEIGHT
  Processed 3880/3880 points
Extracted rows         : 3880
Null height values     : 68
Height min             : 1.02 m
Height max             : 20.40 m
Height mean            : 2.84 m
Unique height values   : 57
TandemX height null count   : 68
tandemx_df height null count   : 0


## Merge SIMARD with EMIT

In [29]:
emit_merged_df_simard = merge_simard_dfs(emit_df, emit_simard_df)

## Merge TANDEMX with EMIT

In [30]:
emit_final_merged_df = merge_tandex_dfs(emit_merged_df_simard, emit_tandemx_df)

In [31]:
print(f"Rows before merge          : {len(emit_merged_df_simard)}")
print(f"Rows after merge           : {len(emit_final_merged_df)}")

print(f"Simard height null count    : {emit_final_merged_df['simard_height_m'].isnull().sum()}")
print(f"TandemX height null count   : {emit_final_merged_df['tandemx_height_m'].isnull().sum()}")

Rows before merge          : 3880
Rows after merge           : 3880
Simard height null count    : 0
TandemX height null count   : 0


In [32]:
assert len(emit_final_merged_df["simard_height_m"].head())
assert len(emit_final_merged_df["tandemx_height_m"].head())

## Save EMIT to CSV

In [34]:
emit_final_merged_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved EMIT output to: {OUTPUT_CSV}")

Saved EMIT output to: ../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EMIT_EO_HEIGHT.csv


# EXTRACT CANOPY DATA FOR SENTINEL-2

In [35]:
INPUT_CSV  = "../../DATA/AGB_DATA/Merged_Data/Sentinel_AGB/AGB_EO_SENTINEL.csv"
OUTPUT_CSV = "../../DATA/AGB_DATA/Merged_Data/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv"

sentinel_df = pd.read_csv(INPUT_CSV)
#print(sentinel_df.columns)
print(f"sentinel_df height null count   : {sentinel_df['height'].isnull().sum()}")

sentinel_df height null count   : 0


## Extract SIMARD for SENTINEL-2

In [36]:
sent_simard_df = extract_simard(sentinel_df)

Total rows    : 8774
Unique plots  : 362
Valid points: 8774 / 8774
EXTRACTING SIMARD CANOPY HEIGHT
  Processed 4500/8774 points
  Processed 8774/8774 points
Extracted rows    : 8774
Null height values: 0
Height min        : 0.00 m
Height max        : 25.00 m
Height mean       : 7.03 m
latitude   longitude 
-2.850264  -40.031187    24
-2.850129  -40.031270    12
-2.849978  -40.031406    19
-2.849830  -40.031500     8
-2.849697  -40.031545    14
-2.849504  -40.031711     7
-2.844100  -40.133954     9
-2.843972  -40.133857    13
-2.843821  -40.133742     8
-2.843674  -40.133709     7
dtype: int64


## Extract TandemX for SENTINEL-2

In [37]:
sent_tandemx_df = extract_tandemx(sentinel_df)

Total rows    : 8774
Unique plots  : 362
Valid points: 8774 / 8774
EXTRACTING TANDEMX CANOPY HEIGHT
  Processed 4500/8774 points
  Processed 8774/8774 points
Extracted rows         : 8774
Null height values     : 807
Height min             : 1.02 m
Height max             : 33.20 m
Height mean            : 8.23 m
Unique height values   : 293
TandemX height null count   : 807
tandemx_df height null count   : 0


## Merge SIMARD with EMIT

In [38]:
sent_merged_df_simard = merge_simard_dfs(sentinel_df, sent_simard_df)

## Merge TANDEMX with EMIT

In [39]:
sent_final_merged_df = merge_tandex_dfs(sent_merged_df_simard, sent_tandemx_df)

In [40]:
print(f"Rows before merge          : {len(sent_merged_df_simard)}")
print(f"Rows after merge           : {len(sent_final_merged_df)}")

print(f"Simard height null count    : {sent_final_merged_df['simard_height_m'].isnull().sum()}")
print(f"TandemX height null count   : {sent_final_merged_df['tandemx_height_m'].isnull().sum()}")

Rows before merge          : 8774
Rows after merge           : 8774
Simard height null count    : 0
TandemX height null count   : 0


In [41]:
assert len(sent_final_merged_df["simard_height_m"].head())
assert len(sent_final_merged_df["tandemx_height_m"].head())

## SAVE SENTINEL-2 to CSV

In [42]:
sent_final_merged_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved EMIT output to: {OUTPUT_CSV}")

Saved EMIT output to: ../../DATA/AGB_DATA/Merged_Data/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv
